# Erlang Hawkes: current state versus filtered Markov approximation

This notebook compares two ways of presenting the same controlled Hawkes problem to each learning algorithm:

- **Filtered**: `(t, X_t, Z_t^1, ..., Z_t^R)`, using a generic bank of exponential readouts of the observed counting process. This is the proposed model-free Markov approximation.
- **Current**: only `(t, X_t)`, with no explicit representation of Hawkes memory.

The two protocols are applied to **CT-DDPG, SAC, and DDPG**. History-window variants and PPO are omitted from the reported experiment. Every filtered learner starts from the same `R=12` exponential readouts; deterministic transformations of those readouts do not add hidden Hawkes state variables. The DGM/HJB oracle knows the exact Erlang Markov state `(t, X_t, Y_t^1, Y_t^2)` and all model parameters. The static comparator is selected independently on validation paths and then frozen before common-path testing.

For filtered CT-DDPG, the actor also learns a zero-initialized three-coefficient residual over the level, decay-rate slope, and curvature of the filter bank. These coefficients are trained by the same continuous-time actor gradient; they are not fitted to the true Erlang kernel and do not use oracle parameters.

The calibrated regime makes memory decision-relevant by combining clustered excitation with a modest jump imprint on `X`. A filtered policy chooses the control from its observed filter state, and that control drives the one observed closed-loop system. No counterfactual system controlled by hidden true memory is used for RL training. Current and filtered variants use the same simulator, objective, and common test paths.

For SAC, the raw filter levels are strongly collinear. Its filtered observation therefore augments the normalized levels with adjacent decay-rate differences, a generic finite-difference approximation to age-weighted exponential memory. This `lifted` representation uses only observed filters and the preselected arithmetic decay grid; it does not use the hidden Erlang phases, the true decay, or fitted kernel weights. SAC checkpoints are evaluated on a reset common-random validation block, and early stopping is disabled for the selected full-budget continuation. The final cost plot keeps the fully tuned baselines; a separate budget plot compares those same test costs against complete training-transition counts. The plotted 90% intervals describe Monte Carlo path uncertainty conditional on one trained policy; a paper-level uncertainty statement should also aggregate independent training seeds.

## Model

This is an **Erlang-2 / phase-type Hawkes kernel** experiment. One jump contributes to intensity through

$$
K(s) \propto \rho s e^{-\rho s}, \qquad s>0.
$$

The true Erlang model is exactly Markovian after adding two auxiliary phases:

$$
\begin{aligned}
dX_t &= (b_0 - \kappa X_t - b_a a_t)\,dt
      + [\sigma_0 + \sigma_x X_t + \sigma_a a_t]_+\,dW_t
      + [\gamma_0 + \gamma_x X_{t-} + \gamma_a a_t]_+\,dN_t,\\
dY_t^1 &= -\rho Y_t^1\,dt + dN_t,\\
dY_t^2 &= (\rho Y_t^1 - \rho Y_t^2)\,dt.
\end{aligned}
$$

The true controlled Hawkes intensity is classical additive and has no link function:

$$
\lambda_t^a = \mu_0 + \mu_x X_{t-} + \mu_a a_t
+ \alpha q(a_t)Y_{t-}^2,
\qquad
q(a)=1-c_a\frac{a}{a_{1/2}+a}.
$$

For an approximate signed readout, positivity is imposed only after aggregating the memory terms:

$$
\lambda_t^{K,a}=\mu(t,X_{t-},a_t)
+\left(\alpha\sum_{r=1}^K c_r(a_t)Z_{t-}^r\right)_+,
\qquad
dZ_t^r=-\eta_rZ_t^r\,dt+dN_t.
$$

Thus approximation coefficients may be negative even though the resulting memory contribution is nonnegative. The DGM/HJB oracle uses the exact known phases `(Y1,Y2)`; filtered CT-DDPG receives only the generic observed filter bank.

The reward contains only quadratic running and terminal costs; there is no event-count penalty:

$$
r_t=-\left(c_xX_t^2+c_a a_t^2\right),
\qquad G(X_T)=-c_TX_T^2.
$$

There is no cap on the total intensity. Only the baseline coefficient and jump amplitude are bounded, exactly as used in Proposition 3.6. The numerical regime uses `mu0=2.00`, `alpha=1.05`, `rho=1.15`, and `gamma0=0.085`. Since the maximal infinite-horizon kernel mass is `alpha/rho=0.913<1`, the calibration is subcritical. The final diagnostic cell checks the finite-horizon true and approximation envelope masses and stress-tests the uncapped simulator.

In [ ]:
# Tunable parameters for this experiment.
# Cached tuned checkpoints load immediately. Set a FORCE_RETRAIN flag after changing parameters.

from dataclasses import asdict
from pathlib import Path
import sys
import json
import csv
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "classical model with positive link", cwd.parent / "classical model with positive link"]
EXP_DIR = next(
    (
        p if p.name == "complex_erlang_experiment" else p / "complex_erlang_experiment"
        for p in candidates
        if (p / "ct_ddpg_erlang.py").exists() or (p / "complex_erlang_experiment" / "ct_ddpg_erlang.py").exists()
    ),
    cwd / "complex_erlang_experiment",
)
PROJECT_ROOT = next(
    (p for p in (EXP_DIR, *EXP_DIR.parents) if (p / "neural_numpy.py").exists() and (p / "replay.py").exists()),
    EXP_DIR.parent,
)

FIG_DIR = EXP_DIR / "figures"
RESULT_DIR = EXP_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def write_history_csv(path, rows):
    if not rows:
        return
    fieldnames = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def observation_label(mode):
    return "filtered" if mode == "filters" else mode


def cached_environment_matches(metadata_paths, expected):
    expected_normalized = json.loads(json.dumps(expected, sort_keys=True))
    for metadata_path in metadata_paths:
        if not metadata_path.exists():
            return False
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        stored = json.loads(json.dumps(metadata.get("environment", {}), sort_keys=True))
        if stored != expected_normalized:
            return False
    return True


for module_path in (PROJECT_ROOT, EXP_DIR, EXP_DIR.parent):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

ENV_PARAMS = dict(
    horizon=5.0,
    dt=0.02,
    beta=0.02,
    mu0=2.00,
    mu_x=0.03,
    mu_a=-0.04,
    excitation=1.05,
    control_effect_strength=1.30,
    action_half_saturation=0.35,
    erlang_decay=1.15,
    baseline_min=1e-6,
    baseline_max=5.0,
    filter_decays=(0.23, 0.46, 0.69, 0.92, 1.15, 1.38, 1.61, 1.84, 2.07, 2.30, 2.53, 2.76),
    drift_constant=0.02,
    drift_reversion=0.45,
    control_drift_strength=0.70,
    diffusion=0.05,
    volatility_x_strength=0.012,
    control_volatility_strength=0.08,
    jump_size=0.085,
    jump_size_x_strength=0.005,
    jump_size_action_strength=-0.010,
    jump_size_min=1e-4,
    jump_size_max=0.20,
    state_cost=0.3000,
    control_cost=0.2250,
    terminal_x_cost=0.2250,
)

CT_OBSERVATION_MODES = ("current", "filters")
CT_TRAINING = dict(
    episodes=140,
    warmup_episodes=30,
    updates_per_episode=10,
    eval_every=20,
    eval_episodes=100,
    checkpoint_seed=4019,
    early_stop_patience=5,
    validation_episodes=240,
    validation_seed=8100,
    seed=19,
    blas_threads=2,
    lr_actor=2e-5,
    lr_actor_profile=0.0,
    lr_critic=6e-4,
    initial_action=0.22,
    actor_t_skip=0.0,
    actor_x_skip=1.2,
    actor_z_skip=0.0,
    actor_z_slope_skip=0.0,
    actor_z_curve_skip=0.0,
    actor_update_every=1,
    tau=0.003,
)

CT_TRAINING_BY_MODE = {
    "filters": dict(
        episodes=140,
        warmup_episodes=30,
        updates_per_episode=10,
        eval_episodes=100,
        early_stop_patience=5,
        lr_actor=5e-6,
        lr_actor_profile=1e-4,
        initial_action=0.16,
        actor_t_skip=0.0,
        actor_x_skip=6.0,
        actor_z_skip=1.0,
        actor_z_slope_skip=0.0,
        actor_z_curve_skip=0.0,
        actor_update_every=1,
        tau=0.003,
    ),
}

DGM_TRAINING = dict(
    steps=650,
    batch_size=384,
    hidden_dim=72,
    num_layers=3,
    lr=1.5e-3,
)

SB3_JOB_CONFIGS = {
    "sac_current": dict(algorithm="sac", observation="current", feature_mode="matched", timesteps=180_000, learning_rate=1e-4, batch_size=512, gradient_steps=1, n_steps=1, tau=0.005, ent_coef="auto_0.005", weight_decay=1e-5, reward_scale=25.0, eval_freq=10_000, eval_episodes=96, early_stop_patience=0, early_stop_min_evals=1),
    "sac_filters": dict(algorithm="sac", observation="filters", feature_mode="lifted", timesteps=220_000, learning_rate=1e-4, batch_size=512, gradient_steps=1, n_steps=1, tau=0.005, ent_coef="auto_0.005", weight_decay=1e-5, reward_scale=25.0, eval_freq=10_000, eval_episodes=96, early_stop_patience=0, early_stop_min_evals=1, warm_start_key="sac_current"),
    "ddpg_current": dict(algorithm="ddpg", observation="current", feature_mode="matched", timesteps=160_000, learning_rate=1e-4, batch_size=512, gradient_steps=1, tau=0.002, noise=0.05, weight_decay=1e-4, reward_scale=25.0, eval_freq=20_000, eval_episodes=64, early_stop_patience=3, early_stop_min_evals=1),
    "ddpg_filters": dict(algorithm="ddpg", observation="filters", feature_mode="matched", timesteps=160_000, learning_rate=1e-4, batch_size=512, gradient_steps=1, tau=0.002, noise=0.05, weight_decay=1e-4, reward_scale=25.0, eval_freq=20_000, eval_episodes=64, early_stop_patience=3, early_stop_min_evals=1, warm_start_key="ddpg_current"),
}

RUN_SB3 = True
FORCE_RETRAIN_CT = False
FORCE_RETRAIN_DGM = False
FORCE_RETRAIN_SB3 = False
CT_PARALLEL_WORKERS = 2
SB3_PARALLEL_WORKERS = 4

VALIDATION_EPISODES = 400
PATH_EPISODES = 20
STATIC_ACTIONS = [0.26]
STATIC_SELECTION_ACTIONS = (0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28)
STATIC_SELECTION_SEED = 1731
STATIC_SELECTION_EPISODES = 1200
STATIC_PROJECTION_STEP = 0.01
FORCE_RESELECT_STATIC = False
SURFACE_GRID_SIZE = 25
SURFACE_MC_EPISODES = 4
SEED = 999

In [ ]:
from erlang_env import ErlangHawkesConfig, ErlangHawkesEnv
from ct_ddpg_erlang import collect_episode, evaluate_agent
from dgm_erlang_oracle import (
    DGMConfig,
    DGMOraclePolicy,
    evaluate_dgm_policy,
    load_dgm_oracle,
    save_dgm_oracle,
    train_dgm_oracle,
)
from plotting_erlang import (
    plot_benchmark_returns,
    plot_ct_history,
    plot_dgm_history,
    plot_episode_ensemble_comparison,
    plot_filtered_training_budget,
    plot_observation_mode_costs,
    plot_policy_mc_cost_surface,
    plot_sb3_training_histories,
    plot_value_surface,
    realized_remaining_cost,
)
from sb3_baselines import collect_sb3_episode, evaluate_sb3_models, sb3_available
from train_tuned_ct_suite import load_tuned_ct_suite, train_tuned_ct_suite
from train_tuned_sb3_suite import load_tuned_suite, make_tuned_jobs, train_tuned_suite
from stability_diagnostics import (
    plot_stability_certificate,
    sufficient_condition_report,
    uncapped_path_diagnostics,
)
from static_control_selection import select_static_control, plot_static_control_selection

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch: {torch.__version__}; DGM device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as exc:
    DEVICE = "cpu"
    print(f"PyTorch check failed, using CPU: {exc}")

if RUN_SB3 and not sb3_available():
    print("stable-baselines3 is not installed in this kernel. Install it with: python -m pip install stable-baselines3")

env_cfg = ErlangHawkesConfig(**ENV_PARAMS)
dgm_cfg = DGMConfig(
    **DGM_TRAINING,
    device=DEVICE,
    print_every=max(1, DGM_TRAINING["steps"] // 10),
)
sb3_jobs = make_tuned_jobs(SB3_JOB_CONFIGS)

filter_obs_dim = 2 + len(env_cfg.filter_decays)
print(f"steps per episode: {env_cfg.steps}; physical state_dim={env_cfg.state_dim}")
print(
    f"observation dimensions: current=2, filtered levels={filter_obs_dim}, "
    f"SAC lifted={filter_obs_dim + len(env_cfg.filter_decays) - 1}"
)
print("Execution: CT-DDPG/SB3 use bounded CPU workers; DGM uses CUDA when available.")

In [ ]:
# Select the constant benchmark on validation paths, independently of final testing.
# A local quadratic smooths Monte Carlo noise before projection to the 0.01 action grid.
static_selection = select_static_control(
    env_cfg,
    evaluate_agent,
    STATIC_SELECTION_ACTIONS,
    seed=STATIC_SELECTION_SEED,
    episodes=STATIC_SELECTION_EPISODES,
    cache_path=RESULT_DIR / "static_control_selection.json",
    environment_signature=asdict(env_cfg),
    projection_step=STATIC_PROJECTION_STEP,
    force=FORCE_RESELECT_STATIC,
)
STATIC_ACTIONS = [float(static_selection["selected_action"])]
print(
    f"Validation-selected static action: a={STATIC_ACTIONS[0]:.2f} "
    f"(continuous quadratic minimizer {static_selection['continuous_minimizer']:.4f})"
)
fig = plot_static_control_selection(
    static_selection,
    FIG_DIR / "static_control_selection.png",
    title="Erlang static-control validation profile",
)
plt.show()

In [ ]:
# Train or load CT-DDPG under current and filtered observations.
ct_metadata_paths = [RESULT_DIR / f"ct_{mode}.json" for mode in CT_OBSERVATION_MODES]
ct_force = FORCE_RETRAIN_CT or not cached_environment_matches(ct_metadata_paths, asdict(env_cfg))
train_tuned_ct_suite(
    RESULT_DIR,
    force=ct_force,
    max_workers=CT_PARALLEL_WORKERS,
    training=CT_TRAINING,
    training_by_mode=CT_TRAINING_BY_MODE,
)
ct_agents, ct_histories, ct_metadata = load_tuned_ct_suite(RESULT_DIR)

for mode in CT_OBSERVATION_MODES:
    history = ct_histories[mode]
    write_history_csv(RESULT_DIR / f"ct_{mode}_history.csv", history)
    fig = plot_ct_history(history, FIG_DIR / f"ct_{mode}_convergence.png")
    plt.show()

# Theory-aligned aliases used by the value surface and kernel diagnostic.
ct_agent = ct_agents["filters"]

In [ ]:
# Train or load the known-parameter DGM/HJB oracle in the exact Erlang state (t, X, Y1, Y2).
dgm_checkpoint = RESULT_DIR / "dgm_oracle.pt"
dgm_metadata_path = RESULT_DIR / "dgm_oracle.json"
dgm_force = FORCE_RETRAIN_DGM or not cached_environment_matches([dgm_metadata_path], asdict(env_cfg))
if dgm_force or not dgm_checkpoint.exists():
    dgm_oracle, dgm_history = train_dgm_oracle(env_cfg, dgm_cfg)
    save_dgm_oracle(dgm_oracle, dgm_checkpoint)
    dgm_policy = DGMOraclePolicy(dgm_oracle)
    dgm_stats = evaluate_dgm_policy(env_cfg, dgm_policy, seed=SEED, episodes=VALIDATION_EPISODES)
    dgm_metadata = {
        "environment": asdict(env_cfg),
        "config": asdict(dgm_cfg),
        "training_history": dgm_history,
        "evaluation": dgm_stats,
    }
    dgm_metadata_path.write_text(json.dumps(dgm_metadata, indent=2), encoding="utf-8")
else:
    dgm_oracle = load_dgm_oracle(dgm_checkpoint, env_cfg=env_cfg, device=DEVICE)
    dgm_metadata = json.loads(dgm_metadata_path.read_text(encoding="utf-8"))
    dgm_history = dgm_metadata["training_history"]
    dgm_policy = DGMOraclePolicy(dgm_oracle)
    print(f"Loaded cached DGM oracle from {dgm_checkpoint}")

# CUDA is used for DGM training; sequential one-state policy queries are faster
# on CPU because GPU launch overhead dominates these tiny batches.
dgm_oracle = dgm_oracle.to("cpu")
dgm_oracle.eval()
dgm_policy = DGMOraclePolicy(dgm_oracle)

fig = plot_dgm_history(dgm_history, FIG_DIR / "dgm_hjb_convergence.png")
plt.show()

In [ ]:
# Train or load SAC/DDPG variants. "New best mean reward" means the validation checkpoint improved.
sb3_models_by_key = {}
sb3_metadata = {}
reuse_cached_sb3_test_rows = False
if RUN_SB3 and sb3_available():
    sb3_metadata_paths = [RESULT_DIR / f"{job.key}.json" for job in sb3_jobs]
    sb3_force = FORCE_RETRAIN_SB3 or not cached_environment_matches(sb3_metadata_paths, asdict(env_cfg))
    train_tuned_suite(
        RESULT_DIR,
        force=sb3_force,
        max_workers=SB3_PARALLEL_WORKERS,
        jobs=sb3_jobs,
    )
    try:
        sb3_models_by_key = load_tuned_suite(RESULT_DIR, jobs=sb3_jobs)
    except ModuleNotFoundError as exc:
        comparison_cache = RESULT_DIR / "comparison.json"
        if sb3_force or not comparison_cache.exists():
            raise
        reuse_cached_sb3_test_rows = True
        print(
            "Could not deserialize the unchanged SB3 checkpoints in this NumPy environment; "
            "reusing their compatible seed-999 test rows from comparison.json."
        )
        print(f"Checkpoint compatibility detail: {exc}")
    sb3_metadata = {
        job.key: json.loads((RESULT_DIR / f"{job.key}.json").read_text(encoding="utf-8"))
        for job in sb3_jobs
    }
    fig = plot_sb3_training_histories(sb3_metadata, FIG_DIR / "sb3_validation_convergence.png")
    plt.show()
else:
    print("Skipping SAC/DDPG baselines in this run.")

In [ ]:
# Evaluate all policies on fresh common Monte Carlo seeds. Cost is -return, so lower is better.
comparison_cache = RESULT_DIR / "comparison.json"
cached_comparison_rows = (
    json.loads(comparison_cache.read_text(encoding="utf-8"))
    if comparison_cache.exists()
    else {}
)
rows = {
    f"ct_ddpg_{mode}": evaluate_agent(
        env_cfg,
        agent,
        seed=SEED,
        episodes=VALIDATION_EPISODES,
    )
    for mode, agent in ct_agents.items()
}
rows["dgm_hjb_oracle"] = evaluate_dgm_policy(
    env_cfg,
    dgm_policy,
    seed=SEED,
    episodes=VALIDATION_EPISODES,
)

for key, (model, mode, history_len, feature_mode) in sb3_models_by_key.items():
    rows[key] = evaluate_sb3_models(
        env_cfg,
        {key: model},
        observation_mode=mode,
        history_len=history_len,
        seed=SEED,
        episodes=VALIDATION_EPISODES,
        feature_mode=feature_mode,
    )[key]

if reuse_cached_sb3_test_rows:
    for key in SB3_JOB_CONFIGS:
        if key not in cached_comparison_rows:
            raise KeyError(f"missing cached common-test statistics for {key}")
        rows[key] = cached_comparison_rows[key]

for a in STATIC_ACTIONS:
    rows[f"constant_{a:.2f}"] = evaluate_agent(
        env_cfg,
        None,
        seed=SEED,
        episodes=VALIDATION_EPISODES,
        constant_action=a,
    )

with (RESULT_DIR / "comparison.json").open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2)

print("Mean discounted cost with 90% Monte Carlo CI:")
for key, stats in sorted(rows.items(), key=lambda item: -item[1]["return_mean"]):
    print(f"{key:>24s}: {-stats['return_mean']:.4f} +/- {stats['return_ci90']:.4f}")

static_key = f"constant_{STATIC_ACTIONS[0]:.2f}"
static_mean = rows[static_key]["return_mean"]
print("\nReturn gain over static control (positive is better):")
for key, stats in rows.items():
    if key != static_key:
        print(f"{key:>24s}: {stats['return_mean'] - static_mean:+.4f}")

fig = plot_benchmark_returns(rows, FIG_DIR / "benchmark_returns.png")
plt.show()

fig = plot_observation_mode_costs(
    rows,
    FIG_DIR / "current_vs_filtered_costs_expanded.png",
)
plt.show()

ct_filtered_training = {**CT_TRAINING, **CT_TRAINING_BY_MODE.get("filters", {})}
filtered_training_transitions = {
    "ct_ddpg_filters": int(ct_filtered_training["episodes"] * env_cfg.steps),
    "sac_filters": int(SB3_JOB_CONFIGS["sac_filters"]["timesteps"]),
    "ddpg_filters": int(SB3_JOB_CONFIGS["ddpg_filters"]["timesteps"]),
}
print("\nFull training-transition budgets for filtered policies:", filtered_training_transitions)
fig = plot_filtered_training_budget(
    rows,
    filtered_training_transitions,
    FIG_DIR / "filtered_cost_vs_training_budget.png",
)
plt.show()

In [ ]:
# Value surfaces use the theory-aligned filtered CT-DDPG only.
# DGM/HJB returns known-parameter cost-to-go U(t, X, Y1, Y2).
# The CT policy surface is a Monte Carlo cost-to-go estimate; the raw critic is a calibration diagnostic.

fig = plot_policy_mc_cost_surface(
    ct_agent,
    env_cfg,
    save_path=FIG_DIR / "ct_ddpg_policy_mc_cost_surface_z0.png",
    title="Filtered CT-DDPG policy Monte Carlo cost surface, Y=Z=0",
    y_level=0.0,
    z_level=0.0,
    grid_size=SURFACE_GRID_SIZE,
    mc_episodes=SURFACE_MC_EPISODES,
    seed=7100,
)
plt.show()

fig = plot_value_surface(
    lambda times, states: -ct_agent.value_path(times, states),
    env_cfg,
    save_path=FIG_DIR / "ct_ddpg_raw_critic_diagnostic_z0.png",
    title="Filtered CT-DDPG critic diagnostic -V(t, X, Z=0)",
    y_level=0.0,
    grid_size=39,
)
plt.show()

fig = plot_value_surface(
    dgm_oracle.value_np,
    env_cfg,
    save_path=FIG_DIR / "dgm_hjb_cost_surface_y0.png",
    title="DGM/HJB oracle cost surface U(t, X, Y1=Y2=0)",
    y_level=0.0,
    grid_size=39,
)
plt.show()


In [ ]:
# Path comparison with 90% confidence intervals and common random-number seeds.
static_action = STATIC_ACTIONS[0]
episode_groups = {
    f"CT-DDPG {observation_label(mode)}": [] for mode in CT_OBSERVATION_MODES
}
episode_groups.update({"DGM/HJB oracle": [], f"static a={static_action:.2f}": []})
for key, (_, mode, _, _) in sb3_models_by_key.items():
    algorithm = key.split("_", 1)[0].upper()
    episode_groups[f"{algorithm} {observation_label(mode)}"] = []
value_path_groups = {key: [] for key in episode_groups}

for i in range(PATH_EPISODES):
    ensemble_seed = 9000 + i
    for mode, agent in ct_agents.items():
        label = f"CT-DDPG {observation_label(mode)}"
        ep = collect_episode(env_cfg, agent, np.random.default_rng(ensemble_seed), exploration_std=0.0)
        episode_groups[label].append(ep)
        value_path_groups[label].append(realized_remaining_cost(ep, env_cfg))

    dgm_ep = collect_episode(env_cfg, dgm_policy, np.random.default_rng(ensemble_seed), exploration_std=0.0)
    static_ep = collect_episode(env_cfg, None, np.random.default_rng(ensemble_seed), constant_action=static_action)
    episode_groups["DGM/HJB oracle"].append(dgm_ep)
    episode_groups[f"static a={static_action:.2f}"].append(static_ep)
    value_path_groups["DGM/HJB oracle"].append(realized_remaining_cost(dgm_ep, env_cfg))
    value_path_groups[f"static a={static_action:.2f}"].append(realized_remaining_cost(static_ep, env_cfg))

    for key, (model, mode, history_len, feature_mode) in sb3_models_by_key.items():
        algorithm = key.split("_", 1)[0].upper()
        label = f"{algorithm} {observation_label(mode)}"
        ep = collect_sb3_episode(
            env_cfg,
            model,
            observation_mode=mode,
            history_len=history_len,
            seed=ensemble_seed,
            feature_mode=feature_mode,
        )
        episode_groups[label].append(ep)
        value_path_groups[label].append(realized_remaining_cost(ep, env_cfg))

fig = plot_episode_ensemble_comparison(
    episode_groups,
    value_path_groups,
    FIG_DIR / "policy_path_ensemble_ci.png",
    ci=0.90,
)
plt.show()


In [ ]:
# Optional: show the latest saved PNGs from disk.
# from IPython.display import Image, display
# for name in [
#     "ct_current_convergence.png",
#     "ct_filters_convergence.png",
#     "dgm_hjb_convergence.png",
#     "sb3_validation_convergence.png",
#     "benchmark_returns.png",
#     "current_vs_filtered_costs_expanded.png",
#     "filtered_cost_vs_training_budget.png",
#     "ct_ddpg_policy_mc_cost_surface_z0.png",
#     "ct_ddpg_raw_critic_diagnostic_z0.png",
#     "dgm_hjb_cost_surface_y0.png",
#     "policy_path_ensemble_ci.png",
#     "ct_kernel_approximation.png",
#     "assumption_3_2_uncapped_diagnostic.png",
# ]:
#     path = FIG_DIR / name
#     if path.exists():
#         print(path)
#         display(Image(filename=str(path)))

In [ ]:
# Script entry points for forced retraining outside the notebook.
# !python train_tuned_ct_suite.py --out-dir results --force --max-workers 2
# !python train_tuned_sb3_suite.py --out-dir results --force --max-workers 4

## Kernel Approximation Diagnostic

This diagnostic separates two relevant signed approximations. The global signed fit approximates the Erlang kernel on a dense deterministic lag grid, so it is the approximation-theory object. The on-policy signed fit regresses the realized Erlang memory on CT-DDPG filter states along simulated paths, so it measures how well the same filter bank represents the memory distribution actually visited by the learned policy. Both are passed through the positive memory link in the approximated intensity.


In [ ]:
import importlib
import kernel_diagnostics_erlang as kde

# Reload so a notebook kernel does not reuse an older diagnostic module.
kde = importlib.reload(kde)

KERNEL_DIAGNOSTIC_EPISODES = 40
fig, kernel_summary = kde.plot_kernel_approximation_diagnostic(
    env_cfg,
    ct_agents["filters"],
    save_path=FIG_DIR / "ct_kernel_approximation.png",
    episodes=KERNEL_DIAGNOSTIC_EPISODES,
    seed=6300,
)
print("global signed kernel-fit weights:", np.round(kernel_summary["global_signed_weights"], 4))
print("on-policy signed memory-fit weights:", np.round(kernel_summary["onpolicy_weights"], 4))
print({k: v for k, v in kernel_summary.items() if "weights" not in k})
plt.show()

# Proposition 3.6 is used only as a transparent sufficient-condition certificate.
# The Monte Carlo panel below does not impose any intensity or event-count cap.
kernel_masses = kde.stability_envelope_masses(env_cfg)
assumption_report = sufficient_condition_report(
    **kernel_masses,
    baseline_bound=env_cfg.baseline_max,
    jump_amplitude_bound=env_cfg.jump_size_max,
    p_star=4.0,
)
uncapped_summary = uncapped_path_diagnostics(
    lambda: ErlangHawkesEnv(env_cfg),
    actions=(0.0, 0.3, 1.0),
    episodes=40,
    seed=8800,
)
print("Assumption 3.2 sufficient-condition report:")
print(json.dumps({k: v for k, v in assumption_report.items()}, indent=2))
for action in ("0.00", "0.30", "1.00"):
    print(action, {k: round(v, 4) for k, v in uncapped_summary[action].items()})
fig = plot_stability_certificate(
    assumption_report,
    uncapped_summary,
    save_path=FIG_DIR / "assumption_3_2_uncapped_diagnostic.png",
    title="Erlang experiment: subcritical certificate and uncapped paths",
)
plt.show()